# 05 - Simplified TransMIL training

**Owner:** Person 3


Train the simplified TransMIL (dimension reduction -> PPEG positional encoding ->
transformer blocks -> CLS token) and inspect the patch-to-patch correlation the
transformer learns. Same split and features as ABMIL and CLAM-SB.


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

In [ ]:
MANIFEST = ROOT / "data/mock/manifest.csv" if (ROOT / "data/mock/manifest.csv").exists() else ROOT / "data/common_split.csv"
FEATURES = MANIFEST.parent / "features"
CONFIG = ROOT / ("configs/transmil_mock.yaml" if "mock" in str(MANIFEST) else "configs/transmil.yaml")
OUT = ROOT / ("results/mock_transmil" if "mock" in str(MANIFEST) else "results/metrics/transmil")
SYNTHETIC = "mock" in str(MANIFEST)
print(MANIFEST, CONFIG, OUT, "synthetic:", SYNTHETIC)

## Model check: shapes and the attention matrix

In [ ]:
import yaml
from src.models import build_model, count_parameters

cfg = yaml.safe_load(CONFIG.read_text())["model"]
cfg["in_dim"] = 1024
model = build_model("transmil", cfg)
x = torch.randn(1, 64, cfg["in_dim"])
out = model(x)
print("logits", tuple(out["logits"].shape), "| attention", tuple(out["attention"].shape))
print("parameters", f"{count_parameters(model):,}")
print("attention sum", float(out["attention"].sum()))

## Train

In [ ]:
import subprocess, sys
seeds = ["0"] if SYNTHETIC else ["0", "1", "2"]
cmd = [sys.executable, "-m", "src.train", "--model", "transmil",
       "--manifest", str(MANIFEST), "--features-root", str(FEATURES),
       "--config", str(CONFIG), "--out-dir", str(OUT), "--seeds", *seeds]
res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-2000:]); print(res.stderr[-800:]); print(res.returncode)

## Attention behaviour (compare with ABMIL / CLAM-SB)

In [ ]:
import numpy as np
with torch.no_grad():
    out = model(torch.randn(1, 400, 1024))
a = out["attention"].squeeze(0).numpy()
print("entropy", float(-(a * np.log(a + 1e-12)).sum()), "| max", float(a.max()), "| top-10 share", float(np.sort(a)[-10:].sum()))

## Notes

* `d_model` must be divisible by `n_heads`.
* Non-square patch counts are zero-padded to a square grid and masked out of attention.
* The CLS->patch attention row is used as the heatmap signal so TransMIL is
  directly comparable with ABMIL/CLAM-SB attention maps.